### Question 1. Install Spark and PySpark

In [ ]:
spark

### Question 2. Yellow November 2025

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("Homework_week6").getOrCreate()

In [ ]:
yellow_2025_11 = spark.read.parquet("yellow_tripdata_2025-11.parquet",header=True, inferSchema=True)

yellow_2025_11.repartition(4).write.parquet("saved_folder")

In [ ]:
import os

folder_path = "saved_folder"

if os.path.exists(folder_path) and os.path.isdir(folder_path):
    print(f"Files in '{folder_path}':")
    for filename in os.listdir(folder_path):
        if filename.endswith('.parquet'):
            file_path = os.path.join(folder_path, filename)
            file_size_bytes = os.path.getsize(file_path)
            file_size_mb = file_size_bytes / (1024 * 1024)
            print(f"- {filename}: {file_size_mb:.2f} MB")
else:
    print(f"Folder '{folder_path}' does not exist or is not a directory.")

### Question 3: Count records

In [ ]:
yellow_2025_11 = yellow_2025_11\
                              .withColumnRenamed("tpep_pickup_datetime", "pickup_datetime")\
                              .withColumnRenamed("tpep_dropoff_datetime", "dropoff_datetime")\
                              .withColumn("pickup_date", F.to_date("pickup_datetime"))

In [ ]:
yellow_2025_11.groupBy("pickup_date").count().filter(F.col("pickup_date") == '2025-11-15').show()

### Question 4: Longest trip

In [ ]:
yellow_2025_11 = yellow_2025_11\
                              .withColumn("trip_hours", 
                               ( 
                                F.unix_timestamp("dropoff_datetime") -
                                F.unix_timestamp("pickup_datetime")
                                ) / 3600)

In [ ]:
yellow_2025_11[['trip_hours']].orderBy(F.col("trip_hours").desc()).first()

### Question 5: User Interface

In [ ]:
http://localhost:4040

### Question 6: Least frequent pickup location zone

In [ ]:
local_zone_df = spark.read.csv("/content/taxi_zone_lookup.csv", header=True, inferSchema=True)

In [ ]:
yellow_2025_11.show()

In [ ]:
local_zone_df = local_zone_df[['LocationID', 'Zone']]

In [ ]:
pickup_zones = local_zone_df.alias("pz")
dropoff_zones = local_zone_df.alias("dz")

yellow = yellow_2025_11.alias("y")

yellow_2025_11_with_zone = (
    yellow
    .join(pickup_zones, F.col("y.PULocationID") == F.col("pz.LocationID"), "inner")
    .withColumnRenamed("Zone", "pickup_zone")
    .join(dropoff_zones, F.col("y.DOLocationID") == F.col("dz.LocationID"), "inner")
    .withColumnRenamed("Zone", "dropoff_zone")
)


In [ ]:
yellow_2025_11_with_zone.groupBy("pickup_zone").count().orderBy(F.col("count").asc()).first()